### Import libraries and dependencies 

In [1]:
import sys
import os

sys.path.append("..")

from utils import get_api_key, get_model_configuration, get_parameter_configuration, LLM

llm = LLM()

### Reflection

- Draft
- Critiqe
- Revise

In [2]:
def draft(query: str) -> dict:
    system_prompt = """
    You are a helpful healthcare assistant providing information about preparing for medical visits.
    Generate a clear, helpful response to the user's question.
    Focus on practical advice and common preparation steps.
    """.strip()

    response = llm.generate(prompt=query, system_prompt=system_prompt)

    return {
        "draft": response["response"],
        "tokens": response["total_tokens"],
        "latency_ms": response["latency_ms"]
    }



In [3]:
def critique(query: str, draft_text: str):
    system_prompt = """
    You are a critical reviewer evaluating healthcare information responses.
    Review the draft and provide constructive feedback on:
        1. Completeness 
        2. Accuracy 
        3. Clarity 
        4. Actionability 
        5. Safety
    Provide specific suggestions for improvement.
    """.strip()

    prompt = f"""
    Original Question: {query}

    Draft Response: {draft_text}

    Please provide a critical review with specific improvement suggestions.
    """.strip()

    response = llm.generate(prompt=prompt, system_prompt=system_prompt)

    return {
        "critique": response["response"],
        "tokens": response["total_tokens"],
        "latency_ms": response["latency_ms"]
    }

In [4]:
def revise(query: str, draft_text: str, critique_text: str):
    system_prompt = """
    You are a healthcare assistant revising your response based on feedback.
    Improve your draft by addressing all critique points.
    Always end with: "This is educational information; verify with the hospital / consult your clinician."
    """.strip()

    prompt = f"""
    Original Question: {query}

    Your Draft: {draft_text}

    Critique Feedback: {critique_text}

    Please provide an improved response that addresses all feedback points.
    """.strip()

    response = llm.generate(prompt=prompt, system_prompt=system_prompt)

    return {
        "revised": response["response"],
        "tokens": response["total_tokens"],
        "latency_ms": response["latency_ms"]
    }

In [5]:
def assess_critique_sufficient(query: str, draft_text: str, critique_text: str):

    system_prompt = """
    You are an evaluator judging whether a critique of a healthcare response indicates
    that the draft is already of sufficient quality and needs no further revision.

    Consider: Does the critique suggest only minor/nitpicky improvements, or does it
    identify substantive gaps (missing info, inaccuracies, clarity issues, safety gaps)?
    If the critique is largely positive and suggests no major improvements, the draft
    is sufficient. If it clearly asks for important additions or corrections, it is not.

    Respond in exactly this format (no other text):
    SUFFICIENT: YES or NO
    REASONING: <one short sentence explaining your judgment>
    """.strip()

    prompt = f"""
    Original question: {query}

    Draft response (excerpt): {draft_text[:500]}{"..." if len(draft_text) > 500 else ""}

    Critique: {critique_text}

    Is the draft already sufficient based on this critique? Use SUFFICIENT and REASONING as specified.
    """.strip()

    response = llm.generate(prompt=prompt, system_prompt=system_prompt)
    text = response["response"].strip().upper()

    sufficient = "SUFFICIENT: YES" in text or "SUFFICIENT:YES" in text
    reasoning = ""

    if "REASONING:" in response["response"]:
        reasoning = response["response"].split("REASONING:")[-1].strip().split("\n")[0]

    
    return {
        "sufficient": sufficient,
        "reasoning": reasoning or ("Critique indicates sufficient quality." if sufficient else "Critique suggests further improvement needed."),
        "tokens": response["total_tokens"],
        "latency_ms": response["latency_ms"]
    }


In [6]:
def reflection_loop(query: str, max_iterations: int = 1, verbose: bool = True) -> dict:
    import time
    start_time = time.time()

    # Stage 1: Initial Draft
    if verbose:
        print("=" * 80)
        print("STAGE 1: INITIAL DRAFT")
        print("=" * 80)

    draft_result = draft(query)
    current_response = draft_result["draft"]
    total_tokens = draft_result["tokens"]

    if verbose:
        print(f"\n{current_response}")
        print(f"\n[Tokens: {draft_result['tokens']}, Latency: {draft_result['latency_ms']}ms]\n")

    # Store all iterations
    iterations = []

    # Stage 2: Iterative Critique -> Revise
    for i in range(max_iterations):
        if verbose:
            print("=" * 80)
            if max_iterations == 1:
                print("STAGE 2: CRITIQUE")
            else:
                print(f"ITERATION {i + 1} of {max_iterations}: CRITIQUE")
            print("=" * 80)
        
        critique_result = critique(query, current_response)
        total_tokens += critique_result["tokens"]
        
        if verbose:
            print(f"\n{critique_result['critique']}")
            print(f"\n[Tokens: {critique_result['tokens']}, Latency: {critique_result['latency_ms']}ms]\n")
        
        # Assess whether critique indicates quality is sufficient (for multi-iteration mode)
        # Uses LLM-based judgment instead of hardcoded phrase matching
        if max_iterations > 1:
            assessment = assess_critique_sufficient(query, current_response, critique_result["critique"])
            total_tokens += assessment.get("tokens", 0)
            if assessment.get("sufficient", False):
                if verbose:
                    print(f"Quality sufficient (assessor: {assessment.get('reasoning', 'N/A')}). Stopping at iteration {i + 1}.")
                iterations.append({
                    "iteration": i + 1,
                    "critique": critique_result,
                    "revised": None,
                    "stopped_early": True,
                    "assessment": assessment
                })
                break
        
        # Stage 3: Revise
        if verbose:
            print("=" * 80)
            if max_iterations == 1:
                print("STAGE 3: REVISE")
            else:
                print(f"ITERATION {i + 1} of {max_iterations}: REVISE")
            print("=" * 80)
        
        revise_result = revise(query, current_response, critique_result["critique"])
        total_tokens += revise_result["tokens"]
        current_response = revise_result["revised"]
        
        if verbose:
            print(f"\n{current_response}")
            print(f"\n[Tokens: {revise_result['tokens']}, Latency: {revise_result['latency_ms']}ms]\n")
        
        iterations.append({
            "iteration": i + 1,
            "critique": critique_result,
            "revised": revise_result,
            "stopped_early": False
        })

    end_time = time.time()
    total_time_ms = int((end_time - start_time) * 1000)

    if verbose:
        print("=" * 80)
        print("SUMMARY")
        print("=" * 80)
        if max_iterations == 1:
            print(f"Total tokens: {total_tokens}")
        else:
            print(f"Initial draft + {len(iterations)} iteration(s)")
            print(f"Total tokens: {total_tokens}")
            print(f"Avg tokens per iteration: {total_tokens // (len(iterations) + 1)}")
        print(f"Total time: {total_time_ms}ms")


    return {
        "query": query,
        "draft": draft_result,
        "iterations": iterations,
        "final_answer": current_response,
        "total_iterations": len(iterations),
        "total_tokens": total_tokens,
        "total_time_ms": total_time_ms
    }

In [7]:
query1 = "How should I prepare for a dermatology OPD visit?"
result1 = reflection_loop(query1, verbose=True)

STAGE 1: INITIAL DRAFT

Preparing for a dermatology outpatient department (OPD) visit can help ensure you make the most of your appointment. Here are some practical steps to consider:

1. **Know Your Medical History**: Be prepared to discuss your medical history, including any past skin issues, allergies, or other health conditions. Consider any family history of skin diseases as well.

2. **List of Medications**: Bring a list of all medications you are currently taking, including over-the-counter drugs, supplements, and any topical treatments for your skin.

3. **Document Symptoms**: Keep a detailed record of your symptoms. Note when they started, how they have changed, and any factors that may worsen or improve the condition (like weather, stress, or specific products).

4. **Skin Care Routine**: Be ready to explain your daily skin care regimen, including any products you use (cleansers, moisturizers, sunscreens, etc.) and any recent changes to it.

5. **Photos**: If possible, take p

In [8]:
result1

{'query': 'How should I prepare for a dermatology OPD visit?',
 'draft': {'draft': 'Preparing for a dermatology outpatient department (OPD) visit can help ensure you make the most of your appointment. Here are some practical steps to consider:\n\n1. **Know Your Medical History**: Be prepared to discuss your medical history, including any past skin issues, allergies, or other health conditions. Consider any family history of skin diseases as well.\n\n2. **List of Medications**: Bring a list of all medications you are currently taking, including over-the-counter drugs, supplements, and any topical treatments for your skin.\n\n3. **Document Symptoms**: Keep a detailed record of your symptoms. Note when they started, how they have changed, and any factors that may worsen or improve the condition (like weather, stress, or specific products).\n\n4. **Skin Care Routine**: Be ready to explain your daily skin care regimen, including any products you use (cleansers, moisturizers, sunscreens, etc

In [9]:
query2 = "What precautions should I take before blood tests?"

# Run with 3 iterations for progressive refinement
result2 = reflection_loop(query2, max_iterations=3, verbose=True)


STAGE 1: INITIAL DRAFT

Before going for blood tests, it’s important to take a few precautions to ensure accurate results and a smooth experience. Here are some helpful steps you can follow:

1. **Fasting Requirements**: Check with your healthcare provider whether you need to fast before the test. Some blood tests, like those measuring glucose or cholesterol levels, often require you to avoid food and drink (except water) for a certain period, usually 8 to 12 hours.

2. **Hydration**: Drink plenty of water before your blood test, unless you have been instructed otherwise. Staying hydrated makes it easier to draw blood and can help improve your blood flow.

3. **Medications**: Inform your healthcare provider about any medications or supplements you are taking, as some may interfere with test results. They may instruct you to skip certain medications prior to the test.

4. **Avoiding Intense Exercise**: Refrain from heavy exercise for 24 hours before the test, as it can alter certain blo